# Ingestion

The `IngestClient` wraps the Prescient Ingest API and returns typed pydantic models. The workflow follows two phases:

1. **Create** an ingestion from a YAML specification, then poll until it is `READY`.
2. **Start** the ingestion, then poll until it is `DONE` (or `FAILED` / `INCOMPLETE`).

After the first ingestion completes, new data can be picked up by creating additional **batches** that follow the same specification.

Authentication is delegated to a `PrescientClient`, so the same `config.env` used elsewhere in the SDK works here. By default the Ingest API base URL is `<prescient_endpoint_url>/ingest/`; set `PRESCIENT_INGEST_ENDPOINT_URL` to override that (e.g. when the Ingest API is deployed at a separate host).

In [ ]:
from pathlib import Path

from prescient_sdk.ingest import IngestClient
from prescient_sdk.ingest_models import Status

client = IngestClient(env_file="config.env")

# Quick liveness check against the Ingest API.
client.health_check()

## Create an ingestion

`create_ingestion` accepts a `Path`, a path string, or raw `bytes` for the YAML specification. The server scans the configured input locations asynchronously, so the returned `Ingestion` will usually start in the `SCANNING` state.

In [ ]:
ingestion = client.create_ingestion(Path("spec.yaml"))
ingestion.id, ingestion.status

## Wait for scanning to finish

`wait_for_status` polls `get_ingestion` until the status reaches one of the target values. The default targets cover all terminal/decision states (`READY`, `DONE`, `FAILED`, `INCOMPLETE`), which is what you want after creating an ingestion — you are waiting for `SCANNING` to resolve.

In [ ]:
ingestion = client.wait_for_status(ingestion.id, poll_interval=5, timeout=300)
ingestion.status

## Inspect what was found

Always check the errors endpoint even on a successful scan — warnings live there too.

In [ ]:
errors = client.get_ingestion_errors(ingestion.id)
for err in errors:
    print(f"[{err.severity.value}] {err.description}")

input_files = client.get_ingestion_input_files(ingestion.id)
output_files = client.get_ingestion_output_files(ingestion.id)
print(f"{len(input_files)} input file(s), {len(output_files)} planned output(s)")

## Start the ingestion

Once the inputs and outputs look right, kick off the actual ingestion. Then poll until it finishes — only success/failure states this time, since `READY` is no longer interesting.

In [ ]:
if ingestion.status is Status.READY:
    client.start_ingestion(ingestion.id)
    ingestion = client.wait_for_status(
        ingestion.id,
        target_statuses=[Status.DONE, Status.FAILED, Status.INCOMPLETE],
        poll_interval=10,
        timeout=3600,
    )
ingestion.status

## Check individual output files

Each `OutputFile` carries its own `FileStatus` so partial failures are visible after the ingestion finishes.

In [ ]:
for out in client.get_ingestion_output_files(ingestion.id):
    print(f"{out.task_name}: {out.status.value} ({out.stac_item_id})")

for err in client.get_ingestion_errors(ingestion.id):
    print(f"[{err.severity.value}] {err.description}")

## Ingesting new data with a new batch

After the initial ingestion is done, you can scan the same input sources for any newly arrived data by creating a new batch. The batch follows the original specification.

In [ ]:
batch = client.create_batch(ingestion.id)
batch = client.wait_for_status(
    ingestion.id,
    batch_number=batch.batch_number,
    poll_interval=5,
    timeout=300,
)

if batch.status is Status.READY:
    client.start_batch(ingestion.id, batch.batch_number)
    batch = client.wait_for_status(
        ingestion.id,
        batch_number=batch.batch_number,
        target_statuses=[Status.DONE, Status.FAILED, Status.INCOMPLETE],
        poll_interval=10,
        timeout=3600,
    )

batch.status

## Uploading the spec from memory

`create_ingestion` also accepts raw `bytes` — useful when the spec is built programmatically rather than read from disk.

In [ ]:
spec_yaml = """
user: example@sparkgeo.com
locations:
  input:
    path: s3://my-bucket/incoming/
source_file_sets:
  rasters:
    location: input
    pattern: "*.tif"
tasks:
  move_main:
    type: move
    input_location: input
"""

ingestion = client.create_ingestion(spec_yaml.encode("utf-8"))
ingestion.id, ingestion.status